# Strategy Analysis Notebook

**Purpose:** Validate the mean-reversion hypothesis — does the strategy have tradeable edge, and under what conditions?

**Input:** `clean.csv` — produced by `preparing.ipynb`. One row per intraday bar, multiple bars per trading day.

**Two DataFrames used throughout:**
- `df` — all bars (used for time-of-day analysis)
- `df_day` — one row per trading day (used for win rate, VIX filter, and most other analyses)

Run cells top-to-bottom. Requires `clean.csv` in the same directory.

---
## Section 0: Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import binomtest

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# ── Color convention (used on every win-rate chart) ──────────────────────────
def win_rate_color(rate):
    """Green >55%, yellow 50-55%, red <50%."""
    if rate > 0.55:
        return '#2ecc71'
    elif rate >= 0.50:
        return '#f1c40f'
    else:
        return '#e74c3c'

def bar_colors(rates):
    return [win_rate_color(r) for r in rates]

def add_50pct_line(ax):
    ax.axhline(0.50, color='black', linestyle='--', linewidth=1, label='50% baseline')

def fmt_pct(ax):
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))


def annotate_heatmap(rate_df, count_df, flag_threshold=None):
    """Build annotated cell strings for seaborn heatmaps.
    Adds N count below each percentage; flags cells with count < flag_threshold with *.
    """
    annot = rate_df.copy().astype(object)
    for r in rate_df.index:
        for c in rate_df.columns:
            val = rate_df.loc[r, c]
            n_val = count_df.loc[r, c]
            if pd.isna(val):
                annot.loc[r, c] = ''
            else:
                flag = '*' if flag_threshold and n_val < flag_threshold else ''
                annot.loc[r, c] = f'{val:.0%}{flag}\n({n_val:.0f})'
    return annot

print('Libraries loaded.')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('clean.csv', parse_dates=['date'])

# Normalize string columns
df['money_made']        = df['money_made'].str.strip().str.lower()
df['philo_result']      = df['philo_result'].str.strip().str.lower()
df['initial_direction'] = df['initial_direction'].str.strip().str.lower()

# vix_favorable may have been saved as string 'True'/'False'
if df['vix_favorable'].dtype == object:
    df['vix_favorable'] = df['vix_favorable'].map({'True': True, 'False': False})

# ── Day-level DataFrame (one row per trading day) ─────────────────────────────
# Day-level columns are identical for every bar within a day; take first row.
DAY_COLS = [
    'date', 'day_of_week', 'year', 'month',
    'market_open_09:30', 'initial_direction',
    'contract_location', 'above', 'below',
    'market_close_16:15', 'market_reg_close_16:00',
    'day_direction_16:15', 'day_direction_reg_16:00',
    'philo_result', 'money_made',
    'diff_open_close', 'rounded_diff_open_close',
    'open_close_diff_sdv_interval', 'open_close_diff_nearest_sdv',
    'diff_open_conLoc', 'open_conLoc_diff_sdv_interval', 'open_conLoc_diff_nearest_sdv',
    'vix_close', 'vix_category', 'vix_favorable',
]
df_day = df[DAY_COLS].groupby('date', as_index=False).first()

# ── Sanity check ─────────────────────────────────────────────────────────────
print(f'Bars loaded      : {len(df):,}')
print(f'Unique days      : {len(df_day):,}')
print(f'Date range       : {df_day["date"].min().date()} → {df_day["date"].max().date()}')
print(f'Avg bars/day     : {len(df) / len(df_day):.1f}')
print()

# Missing values across key columns
key_cols = ['money_made', 'philo_result', 'vix_close', 'vix_favorable',
            'contract_location', 'diff_open_close', 'open_close_diff_sdv_interval']
missing = df_day[key_cols].isna().sum()
if missing.any():
    print('Missing values in day-level data:')
    print(missing[missing > 0])
else:
    print('No missing values in key columns.')

df_day.head(3)

---
## Section 1: Dataset Overview

Before trusting any win rate, confirm the dataset is complete and balanced across years. A suspiciously sparse year (data gap) or an unbalanced dataset could skew all downstream results.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Trading days per year
days_per_year = df_day.groupby('year').size()
axes[0].bar(days_per_year.index.astype(str), days_per_year.values, color='steelblue')
for i, (yr, n) in enumerate(days_per_year.items()):
    axes[0].text(i, n + 1, str(n), ha='center', va='bottom', fontsize=9)
axes[0].set_title('Trading Days per Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Days')

# Bars per day distribution
bars_per_day = df.groupby('date').size()
axes[1].hist(bars_per_day.values, bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Distribution: Bars per Trading Day')
axes[1].set_xlabel('Bar count')
axes[1].set_ylabel('Days')
axes[1].axvline(bars_per_day.median(), color='black', linestyle='--', label=f'Median={bars_per_day.median():.0f}')
axes[1].legend(fontsize=9)

# Initial direction split
dir_counts = df_day['initial_direction'].value_counts()
axes[2].bar(dir_counts.index, dir_counts.values, color=['#e74c3c', '#2ecc71', '#95a5a6'])
for i, (lbl, n) in enumerate(dir_counts.items()):
    axes[2].text(i, n + 1, str(n), ha='center', va='bottom', fontsize=9)
axes[2].set_title('Initial Direction Split')
axes[2].set_ylabel('Days')

plt.tight_layout()
plt.show()

# VIX category distribution
print('\nVIX category distribution (days):')
print(df_day['vix_category'].value_counts().to_frame('count').assign(
    pct=lambda x: (x['count'] / x['count'].sum() * 100).round(1)
))

*Interpretation: note any years with significantly fewer days (possible data gaps), and whether above/below days are roughly balanced. An imbalanced direction split could indicate a systematic bias in the data.*

---
## Section 2: Core Hypothesis — Overall Win Rate ⭐

The fundamental question: does the mean-reversion strategy win more than 50% of the time?

We track two metrics:
- **`philo_result`** — did the reversal actually happen? (direction-only)
- **`money_made`** — did price cross the contract strike? (the tradeable outcome)

The gap between them reveals the cost of the strike placement.

In [ ]:
df_core = df_day.dropna(subset=['money_made', 'philo_result'])
n = len(df_core)

philo_wins  = (df_core['philo_result'] == 'correct').sum()
money_wins  = (df_core['money_made']   == 'yes').sum()
philo_rate  = philo_wins / n
money_rate  = money_wins / n

# Binomial tests vs. p=0.50
philo_test = binomtest(philo_wins, n, p=0.5, alternative='greater')
money_test = binomtest(money_wins, n, p=0.5, alternative='greater')

# 95% Clopper-Pearson confidence intervals
philo_ci = philo_test.proportion_ci(confidence_level=0.95)
money_ci = money_test.proportion_ci(confidence_level=0.95)

print('=' * 58)
print(f'  N = {n:,} trading days')
print('=' * 58)
print(f'  Philosophical win rate  : {philo_rate:.1%}  (p={philo_test.pvalue:.4f})')
print(f'    95% CI: [{philo_ci.low:.1%}, {philo_ci.high:.1%}]')
print()
print(f'  Tradeable win rate      : {money_rate:.1%}  (p={money_test.pvalue:.4f})')
print(f'    95% CI: [{money_ci.low:.1%}, {money_ci.high:.1%}]')
print()
print(f'  Strike cost (gap)       : {philo_rate - money_rate:.1%}')
print('=' * 58)
print()

if money_test.pvalue < 0.05:
    print('  RESULT: Tradeable win rate is statistically significant (p < 0.05). Strategy shows edge.')
elif money_test.pvalue < 0.10:
    print('  RESULT: Marginal significance (0.05 < p < 0.10). More data needed for confidence.')
else:
    print('  RESULT: Not statistically significant (p >= 0.10). Cannot reject null hypothesis.')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

labels = ['Philosophical\n(direction)', 'Tradeable\n(crossed strike)']
rates  = [philo_rate, money_rate]
xerr_lo = [philo_rate - philo_ci.low,  money_rate - money_ci.low]
xerr_hi = [philo_ci.high - philo_rate, money_ci.high - money_rate]

bars = ax.barh(labels, rates, color=bar_colors(rates), height=0.5)
ax.errorbar(rates, labels, xerr=[xerr_lo, xerr_hi],
            fmt='none', color='black', capsize=5, linewidth=1.5)
ax.axvline(0.50, color='black', linestyle='--', linewidth=1)
for bar, rate in zip(bars, rates):
    ax.text(rate + 0.005, bar.get_y() + bar.get_height()/2,
            f'{rate:.1%}', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0.40, min(max(rates) + 0.10, 1.0))
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_title('Overall Win Rate vs. 50% Baseline', fontsize=13)
ax.set_xlabel('Win Rate')
plt.tight_layout()
plt.show()

---
## Section 3: VIX Filter Validation ⭐

Does restricting trades to days where VIX is 15–30 (`vix_favorable == True`) improve win rate? This was the key question from the VIX integration work. We also need to know: how often does this filter apply — if it covers only 10% of days, it may not be worth using.

VIX categories: Low (<15) · Moderate (15–25) · Elevated (25–35) · High (>35). `vix_favorable` covers Moderate + the lower part of Elevated (VIX 15–30).

In [ ]:
# ── Favorable vs. unfavorable ─────────────────────────────────────────────────
fav   = df_day[df_day['vix_favorable'] == True]
unfav = df_day[df_day['vix_favorable'] == False]

def win_rate_stats(subset, label):
    n_s = len(subset.dropna(subset=['money_made']))
    wins = (subset['money_made'] == 'yes').sum()
    rate = wins / n_s if n_s > 0 else np.nan
    pct_of_total = n_s / len(df_day) * 100
    return {'Label': label, 'N days': n_s, '% of total': f'{pct_of_total:.1f}%', 'Win rate': f'{rate:.1%}'}

summary = pd.DataFrame([
    win_rate_stats(df_day, 'All days'),
    win_rate_stats(fav,    'VIX favorable (15–30)'),
    win_rate_stats(unfav,  'VIX unfavorable'),
])
print(summary.to_string(index=False))

In [ ]:
# ── Win rate by VIX category ──────────────────────────────────────────────────
cat_order = ['Low', 'Moderate', 'Elevated', 'High']
vix_stats = (
    df_day.dropna(subset=['money_made', 'vix_category'])
    .groupby('vix_category')
    .agg(n=('money_made', 'count'),
         wins=('money_made', lambda x: (x == 'yes').sum()))
    .assign(win_rate=lambda x: x['wins'] / x['n'])
    .reindex([c for c in cat_order if c in df_day['vix_category'].unique()])
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: favorable vs unfavorable
fav_rates  = [(fav['money_made'] == 'yes').mean(), (unfav['money_made'] == 'yes').mean()]
fav_labels = [f'Favorable\n(VIX 15–30)\nN={len(fav)}', f'Unfavorable\nN={len(unfav)}']
bars_l = axes[0].bar(fav_labels, fav_rates, color=bar_colors(fav_rates), width=0.4)
add_50pct_line(axes[0])
fmt_pct(axes[0])
for bar, rate in zip(bars_l, fav_rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, rate + 0.004,
                 f'{rate:.1%}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Win Rate: VIX Favorable vs. Unfavorable')
axes[0].set_ylim(0, max(fav_rates) + 0.12)

# Right: by VIX category
cats  = vix_stats.index.tolist()
rates = vix_stats['win_rate'].tolist()
ns    = vix_stats['n'].tolist()
bars_r = axes[1].bar(cats, rates, color=bar_colors(rates))
add_50pct_line(axes[1])
fmt_pct(axes[1])
for bar, rate, n_cat in zip(bars_r, rates, ns):
    flag = '*' if n_cat < 30 else ''
    axes[1].text(bar.get_x() + bar.get_width()/2, rate + 0.004,
                 f'{rate:.1%}\n(N={n_cat}{flag})', ha='center', va='bottom', fontsize=9)
axes[1].set_title('Win Rate by VIX Category')
axes[1].set_ylim(0, max(rates) + 0.15)
axes[1].set_xlabel('* = fewer than 30 observations')

plt.suptitle('VIX Filter Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

*Interpretation: Compare the favorable vs. unfavorable bars — does filtering to VIX 15–30 meaningfully improve win rate? Also check the category breakdown: if 'Moderate' and 'Elevated' show the highest rates, the 15–30 threshold is validated. If 'Low' (calm markets) or 'High' (panic) also perform well, the threshold may need adjustment.*

---
## Section 4: Time-of-Day Analysis

Which intraday entry times have the highest win rates? This uses the full bar-level `df` — each row represents a potential entry at a specific time. Early bars capture the initial move; later bars may reflect a completed reversal.

In [ ]:
MIN_BARS = 50  # flag unreliable time slots

time_stats = (
    df.dropna(subset=['money_made', 'time'])
    .groupby('time')
    .agg(n=('money_made', 'count'),
         wins=('money_made', lambda x: (x == 'yes').sum()))
    .assign(win_rate=lambda x: x['wins'] / x['n'])
    .sort_index()
)

fig, ax1 = plt.subplots(figsize=(16, 5))

colors = bar_colors(time_stats['win_rate'].tolist())
ax1.plot(time_stats.index, time_stats['win_rate'], color='steelblue',
         linewidth=2, zorder=3, label='Win rate')
ax1.scatter(time_stats.index, time_stats['win_rate'], color=colors, s=40, zorder=4)
add_50pct_line(ax1)
fmt_pct(ax1)
ax1.set_ylabel('Win Rate', color='steelblue')
ax1.tick_params(axis='x', rotation=45, labelsize=7)

ax2 = ax1.twinx()
ax2.bar(time_stats.index, time_stats['n'], alpha=0.25, color='grey', label='Bar count')
ax2.set_ylabel('Bar Count', color='grey')
ax2.axhline(MIN_BARS, color='grey', linestyle=':', linewidth=1)

ax1.set_title('Win Rate and Bar Count by Time of Day', fontsize=13)
fig.legend(loc='upper right', bbox_to_anchor=(0.98, 0.95))
plt.tight_layout()
plt.show()

reliable = time_stats[time_stats['n'] >= MIN_BARS].sort_values('win_rate', ascending=False)
print(f'\nTop 10 time slots (N ≥ {MIN_BARS}):')
print(reliable.head(10)[['n', 'win_rate']].to_string())
print(f'\nBottom 10 time slots (N ≥ {MIN_BARS}):')
print(reliable.tail(10)[['n', 'win_rate']].to_string())

*Interpretation: Look for a time window where the line stays consistently above 50%. Dots below the bar chart's dotted line have fewer than 50 observations — treat those rates with caution. High win-rate times with high bar counts are the most actionable findings.*

---
## Section 5: Day-of-Week Analysis

Does the strategy perform consistently across all weekdays? Mondays and Fridays often have different volatility profiles (gap risk on Monday, position squaring on Friday).

In [ ]:
DOW_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

dow_stats = (
    df_day.dropna(subset=['money_made'])
    .groupby('day_of_week')
    .agg(n=('money_made', 'count'),
         wins=('money_made', lambda x: (x == 'yes').sum()))
    .assign(win_rate=lambda x: x['wins'] / x['n'])
    .reindex([d for d in DOW_ORDER if d in df_day['day_of_week'].unique()])
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: win rate by day of week
days   = dow_stats.index.tolist()
rates  = dow_stats['win_rate'].tolist()
ns     = dow_stats['n'].tolist()
bars = axes[0].bar(days, rates, color=bar_colors(rates))
add_50pct_line(axes[0])
fmt_pct(axes[0])
for bar, rate, n in zip(bars, rates, ns):
    axes[0].text(bar.get_x() + bar.get_width()/2, rate + 0.003,
                 f'{rate:.1%}\n(N={n})', ha='center', va='bottom', fontsize=9)
axes[0].set_title('Win Rate by Day of Week')
axes[0].set_ylim(0, max(rates) + 0.13)

# Right: heatmap — day_of_week x vix_category
cat_order = ['Low', 'Moderate', 'Elevated', 'High']
pivot = (
    df_day.dropna(subset=['money_made', 'vix_category'])
    .groupby(['day_of_week', 'vix_category'])
    .agg(n=('money_made', 'count'),
         win_rate=('money_made', lambda x: (x == 'yes').mean()))
    .reset_index()
)
heatmap_rate = pivot.pivot(index='day_of_week', columns='vix_category', values='win_rate')
heatmap_n    = pivot.pivot(index='day_of_week', columns='vix_category', values='n')

# Reorder
heatmap_rate = heatmap_rate.reindex(index=[d for d in DOW_ORDER if d in heatmap_rate.index],
                                     columns=[c for c in cat_order if c in heatmap_rate.columns])
heatmap_n    = heatmap_n.reindex(   index=heatmap_rate.index, columns=heatmap_rate.columns)

# Mask cells with N < 30
annot = annotate_heatmap(heatmap_rate, heatmap_n, flag_threshold=30)

sns.heatmap(heatmap_rate, ax=axes[1], cmap='RdYlGn', center=0.5, vmin=0.35, vmax=0.70,
            annot=annot, fmt='', linewidths=0.5, cbar_kws={'format': mticker.PercentFormatter(xmax=1)})
axes[1].set_title('Win Rate: Day × VIX Category\n(* = N < 30, treat with caution)')
axes[1].set_xlabel('VIX Category')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

*Interpretation: If one or two days consistently outperform, a simple day-of-week filter could be added with no additional infrastructure. The heatmap reveals whether the VIX effect is consistent across all days or concentrated on specific days.*

---
## Section 6: Volatility Environment

Does the magnitude of the day's price range affect strategy performance? On low-volatility days, price may not move far enough to cross the contract strike. On extreme days, trending behavior may override mean-reversion.

In [ ]:
SDV_ORDER = ['Extremely Low', 'Low', 'Slightly Low', 'Slightly High', 'High', 'Extremely High']

def sdv_win_rate(col, label):
    stats = (
        df_day.dropna(subset=['money_made', col])
        .groupby(col)
        .agg(n=('money_made', 'count'),
             wins=('money_made', lambda x: (x == 'yes').sum()))
        .assign(win_rate=lambda x: x['wins'] / x['n'])
        .reindex([s for s in SDV_ORDER if s in df_day[col].dropna().unique()])
    )
    return stats

range_stats  = sdv_win_rate('open_close_diff_sdv_interval', 'Daily Range Band')
strike_stats = sdv_win_rate('open_conLoc_diff_sdv_interval', 'Strike Distance Band')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def plot_sdv_bars(ax, stats, title):
    cats  = stats.index.tolist()
    rates = stats['win_rate'].tolist()
    ns    = stats['n'].tolist()
    bars  = ax.bar(range(len(cats)), rates, color=bar_colors(rates), tick_label=cats)
    add_50pct_line(ax)
    fmt_pct(ax)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    for i, (rate, n) in enumerate(zip(rates, ns)):
        flag = '*' if n < 30 else ''
        ax.text(i, rate + 0.003, f'{rate:.1%}\n(N={n}{flag})', ha='center', va='bottom', fontsize=8)
    ax.set_title(title)
    ax.set_ylim(0, max(rates) + 0.15 if rates else 0.7)

plot_sdv_bars(axes[0], range_stats,  'Win Rate by Daily Range Band')
plot_sdv_bars(axes[1], strike_stats, 'Win Rate by Strike Distance Band')

# Box plot: diff_open_close by money_made
df_box = df_day.dropna(subset=['money_made', 'diff_open_close'])
axes[2].boxplot(
    [df_box[df_box['money_made'] == 'yes']['diff_open_close'].values,
     df_box[df_box['money_made'] == 'no']['diff_open_close'].values],
    labels=['Win (yes)', 'Loss (no)'],
    patch_artist=True,
    boxprops=dict(facecolor='#2ecc7155'),
    medianprops=dict(color='black', linewidth=2)
)
axes[2].set_title('Daily Range Distribution:\nWin vs. Loss Days')
axes[2].set_ylabel('diff_open_close (points)')

plt.tight_layout()
plt.show()

*Interpretation: If low-range days have poor win rates, a minimum volatility filter could be added. If farther strikes (high `open_conLoc_diff_sdv_interval`) consistently underperform, strike selection logic should be tightened. The box plot reveals whether winning days tend to have larger ranges (price had room to move to the strike).*

---
## Section 7: Trend Over Time

Is the strategy stable, or is it deteriorating? A strategy that worked in 2018 but not in 2023 is not tradeable today. The rolling 90-day win rate is the most important chart for assessing drift.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Win rate by year
yr_stats = (
    df_day.dropna(subset=['money_made'])
    .groupby('year')
    .agg(n=('money_made', 'count'),
         wins=('money_made', lambda x: (x == 'yes').sum()))
    .assign(win_rate=lambda x: x['wins'] / x['n'])
)
bars_y = axes[0].bar(yr_stats.index.astype(str), yr_stats['win_rate'],
                      color=bar_colors(yr_stats['win_rate'].tolist()))
add_50pct_line(axes[0])
fmt_pct(axes[0])
for bar, (yr, row) in zip(bars_y, yr_stats.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, row.win_rate + 0.003,
                 f'{row.win_rate:.1%}\n(N={row.n:.0f})', ha='center', va='bottom', fontsize=8)
axes[0].set_title('Win Rate by Year')
axes[0].set_ylim(0, yr_stats['win_rate'].max() + 0.14)
axes[0].tick_params(axis='x', rotation=45)

# Win rate by month
mo_stats = (
    df_day.dropna(subset=['money_made'])
    .groupby('month')
    .agg(n=('money_made', 'count'),
         wins=('money_made', lambda x: (x == 'yes').sum()))
    .assign(win_rate=lambda x: x['wins'] / x['n'])
)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
bars_m = axes[1].bar(
    mo_stats.index.map(lambda m: month_labels[m-1]),
    mo_stats['win_rate'],
    color=bar_colors(mo_stats['win_rate'].tolist())
)
add_50pct_line(axes[1])
fmt_pct(axes[1])
for bar, (mo, row) in zip(bars_m, mo_stats.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2, row.win_rate + 0.003,
                 f'{row.win_rate:.1%}', ha='center', va='bottom', fontsize=8)
axes[1].set_title('Win Rate by Month (Seasonality)')
axes[1].set_ylim(0, mo_stats['win_rate'].max() + 0.12)

# Rolling 90-day win rate
daily_sorted = df_day.dropna(subset=['money_made']).sort_values('date').copy()
daily_sorted['win_binary'] = (daily_sorted['money_made'] == 'yes').astype(int)
daily_sorted['rolling_90'] = daily_sorted['win_binary'].rolling(90, min_periods=45).mean()

axes[2].plot(daily_sorted['date'], daily_sorted['rolling_90'],
             color='steelblue', linewidth=1.5, label='90-day rolling win rate')
axes[2].axhline(0.50, color='black', linestyle='--', linewidth=1, label='50% baseline')
axes[2].axhline(daily_sorted['win_binary'].mean(), color='grey',
                linestyle=':', linewidth=1, label='Overall average')
axes[2].fill_between(daily_sorted['date'], 0.50, daily_sorted['rolling_90'],
                      where=daily_sorted['rolling_90'] >= 0.50,
                      alpha=0.15, color='#2ecc71')
axes[2].fill_between(daily_sorted['date'], 0.50, daily_sorted['rolling_90'],
                      where=daily_sorted['rolling_90'] < 0.50,
                      alpha=0.15, color='#e74c3c')
fmt_pct(axes[2])
axes[2].set_title('Rolling 90-Day Win Rate')
axes[2].legend(fontsize=8)
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

*Interpretation: A flat or gently rising rolling line is a green flag — the strategy is stable. A sharply declining line suggests the edge is eroding. If the line dips below 50% for extended periods, identify what was happening in the market at that time (e.g., 2020 COVID volatility, 2022 rate-hike trend).*

---
## Section 8: Initial Direction Analysis

Is the strategy symmetric — does it perform equally well when price opens above vs. below the reference? If one direction is significantly stronger, we might only trade that direction.

In [ ]:
dir_stats = (
    df_day.dropna(subset=['money_made', 'initial_direction'])
    .groupby('initial_direction')
    .agg(n=('money_made', 'count'),
         wins=('money_made', lambda x: (x == 'yes').sum()))
    .assign(win_rate=lambda x: x['wins'] / x['n'])
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: win rate by initial direction
dirs  = dir_stats.index.tolist()
rates = dir_stats['win_rate'].tolist()
ns    = dir_stats['n'].tolist()
bars = axes[0].bar(dirs, rates, color=bar_colors(rates), width=0.4)
add_50pct_line(axes[0])
fmt_pct(axes[0])
for bar, rate, n in zip(bars, rates, ns):
    axes[0].text(bar.get_x() + bar.get_width()/2, rate + 0.003,
                 f'{rate:.1%}\n(N={n})', ha='center', va='bottom')
axes[0].set_title('Win Rate by Initial Direction')
axes[0].set_ylim(0, max(rates) + 0.12)

# Right: 2x2 — direction x vix_favorable
cross = (
    df_day.dropna(subset=['money_made', 'initial_direction', 'vix_favorable'])
    .groupby(['initial_direction', 'vix_favorable'])
    .agg(n=('money_made', 'count'),
         win_rate=('money_made', lambda x: (x == 'yes').mean()))
    .reset_index()
)
pivot_cross = cross.pivot(index='initial_direction', columns='vix_favorable',
                           values='win_rate')
pivot_n     = cross.pivot(index='initial_direction', columns='vix_favorable',
                           values='n')

annot_cross = annotate_heatmap(pivot_cross, pivot_n)

sns.heatmap(pivot_cross, ax=axes[1], cmap='RdYlGn', center=0.5, vmin=0.35, vmax=0.70,
            annot=annot_cross, fmt='', linewidths=0.5,
            cbar_kws={'format': mticker.PercentFormatter(xmax=1)})
axes[1].set_title('Win Rate: Direction × VIX Favorable')
axes[1].set_xlabel('VIX Favorable')
axes[1].set_ylabel('Initial Direction')

plt.tight_layout()
plt.show()

*Interpretation: If 'above' (sell trade) and 'below' (buy trade) show similar win rates, the strategy is symmetric and both directions are valid. A large gap suggests only trading the stronger direction. The heatmap shows whether the VIX filter helps both directions equally or is more valuable for one.*

---
## Section 9: Win/Loss Streak Analysis

Are wins and losses independent, or do they cluster into streaks? Heavy loss clustering is a risk management concern. If the win rate drops significantly after a long win streak, that is a signal to reduce size.

In [ ]:
daily_s = df_day.sort_values('date').dropna(subset=['money_made', 'overall_win_streak', 'overall_loss_streak'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Streak length distributions
max_win  = int(daily_s['overall_win_streak'].max())
max_loss = int(daily_s['overall_loss_streak'].max())

# Count how many times each streak length was the PEAK of a run
# (i.e., where the next value resets to 0)
def streak_peaks(col):
    s = daily_s[col]
    peaks = s[(s > 0) & (s.shift(-1).fillna(0) == 0)]
    return peaks.value_counts().sort_index()

win_peaks  = streak_peaks('overall_win_streak')
loss_peaks = streak_peaks('overall_loss_streak')

axes[0].bar(win_peaks.index, win_peaks.values, color='#2ecc71', edgecolor='white')
axes[0].set_title(f'Win Streak Lengths (max={max_win})')
axes[0].set_xlabel('Consecutive wins')
axes[0].set_ylabel('Frequency')

axes[1].bar(loss_peaks.index, loss_peaks.values, color='#e74c3c', edgecolor='white')
axes[1].set_title(f'Loss Streak Lengths (max={max_loss})')
axes[1].set_xlabel('Consecutive losses')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Win rate following N-game streaks
print('Win rate on the NEXT day, following a win/loss streak of length N:')
print('(Uses overall_win_streak and overall_loss_streak from preparing.ipynb)\n')

# Compute next calendar day's outcome for every row BEFORE filtering.
# shift(-1) on the full sorted daily_s gives the actual next trading day,
# not the next row that happens to match the same streak length.
daily_s = daily_s.copy()
daily_s['next_outcome'] = daily_s['money_made'].shift(-1)

rows = []
for n_streak in range(1, 8):
    after_win  = daily_s.loc[daily_s['overall_win_streak']  == n_streak, 'next_outcome']
    after_loss = daily_s.loc[daily_s['overall_loss_streak'] == n_streak, 'next_outcome']
    wr_after_win  = (after_win  == 'yes').sum() / after_win.notna().sum()  if after_win.notna().sum() > 0 else np.nan
    wr_after_loss = (after_loss == 'yes').sum() / after_loss.notna().sum() if after_loss.notna().sum() > 0 else np.nan
    rows.append({
        'Streak N': n_streak,
        'After win streak (N)': f'{wr_after_win:.1%}' if not np.isnan(wr_after_win) else 'N/A',
        'Sample (wins)': after_win.notna().sum(),
        'After loss streak (N)': f'{wr_after_loss:.1%}' if not np.isnan(wr_after_loss) else 'N/A',
        'Sample (losses)': after_loss.notna().sum(),
    })

streak_table = pd.DataFrame(rows)
print(streak_table.to_string(index=False))

*Interpretation: If loss streaks rarely exceed 4–5 in a row, a drawdown of 5 losses is a normal event — not a signal to stop trading. If the win rate after a 5-game win streak is noticeably lower than the baseline, consider reducing position size. Small sample sizes (N < 20) make these streak-conditional rates unreliable.*

---
## Section 10: Combined Best-Case Conditions

Combine the top filters from earlier sections to find the highest-confidence setup. Then check: how often do those conditions actually occur? A 70% win rate that fires 8 times per year is not very useful.

In [ ]:
# ── Edit these filters based on findings in Sections 3–8 ─────────────────────
# These are starting defaults that can be tuned after reviewing the above sections.

BEST_VIX_FAVORABLE  = True                         # Section 3 finding
BEST_DOW            = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']  # update after Section 5
BEST_SDV_BANDS      = ['Slightly Low', 'Slightly High', 'High', 'Extremely High']  # update after Section 6
BEST_DIRECTION      = ['above', 'below']           # update after Section 8 if asymmetric

# ── Apply combined filter ─────────────────────────────────────────────────────
mask_combined = (
    (df_day['vix_favorable'] == BEST_VIX_FAVORABLE) &
    (df_day['day_of_week'].isin(BEST_DOW)) &
    (df_day['open_close_diff_sdv_interval'].isin(BEST_SDV_BANDS)) &
    (df_day['initial_direction'].isin(BEST_DIRECTION))
)

best = df_day[mask_combined].dropna(subset=['money_made'])
best_rate = (best['money_made'] == 'yes').mean()
years_span = (df_day['date'].max() - df_day['date'].min()).days / 365.25

print('=' * 55)
print(f'  Combined filter: N = {len(best):,} days')
print(f'  Win rate        : {best_rate:.1%}')
print(f'  % of all days   : {len(best)/len(df_day)*100:.1f}%')
print(f'  Est. trades/yr  : {len(best)/years_span:.0f}')
print('=' * 55)

In [ ]:
# ── Sensitivity table — vary VIX and DOW filter ───────────────────────────────
results = []
for vix_fav in [True, False, None]:    # None = no VIX filter
    for sdv_tight in [True, False]:    # True = only 'Slightly High', 'High', 'Extremely High'
        vix_mask = (
            df_day['vix_favorable'] == vix_fav if vix_fav is not None
            else pd.Series(True, index=df_day.index)
        )
        sdv_vals = (['Slightly High', 'High', 'Extremely High'] if sdv_tight
                    else ['Slightly Low', 'Slightly High', 'High', 'Extremely High', 'Low'])
        sdv_mask = df_day['open_close_diff_sdv_interval'].isin(sdv_vals)
        subset   = df_day[vix_mask & sdv_mask].dropna(subset=['money_made'])
        rate     = (subset['money_made'] == 'yes').mean()
        label_vix = ('VIX fav' if vix_fav is True else
                     ('VIX unfav' if vix_fav is False else 'No VIX filter'))
        label_sdv = 'SDV tight' if sdv_tight else 'SDV broad'
        results.append({
            'VIX filter': label_vix,
            'Volatility filter': label_sdv,
            'N days': len(subset),
            'Win rate': f'{rate:.1%}',
            'Est. trades/yr': f'{len(subset)/years_span:.0f}',
        })

sens_df = pd.DataFrame(results)
print('Sensitivity Table — Win Rate vs. Filter Tightness')
print(sens_df.to_string(index=False))

*Interpretation: The sensitivity table shows the quality vs. quantity tradeoff. Look for the row with the best win rate that still produces a reasonable number of annual trades. A combined filter should ideally produce 50+ trades per year to be statistically useful going forward.*

---
## Section 11: Summary & Conclusions

*Fill in after running all sections above.*

### Findings

| Section | Question | Finding |
|---|---|---|
| 2 | Does the strategy beat 50%? | *(fill in: overall win rate, p-value)* |
| 3 | Does VIX filter improve win rate? | *(fill in: rate favorable vs. unfavorable)* |
| 4 | Best time-of-day window? | *(fill in: top time slots)* |
| 5 | Best days of week? | *(fill in: top 2 days)* |
| 6 | Optimal volatility range? | *(fill in: best SDV bands)* |
| 7 | Is strategy stable over time? | *(fill in: rolling win rate trend)* |
| 8 | Is strategy symmetric by direction? | *(fill in: above vs. below rates)* |
| 9 | Worst streak risk? | *(fill in: max loss streak length)* |
| 10 | Best combined win rate? | *(fill in: combined rate, trades/yr)* |

---

### Recommendation

*(Fill in: go / conditional go / no-go, with the specific filter conditions)*

---

### What This Analysis Cannot Answer

- **Transaction costs** — Nadex binary options have a fee per contract; this must be modeled against the win rate to determine net profitability
- **Slippage** — Contract fills may not be at the exact strike level; real-world fills could differ
- **Contract expiry mechanics** — The binary option expires at a fixed time; end-of-day close may not equal the expiry price
- **Out-of-sample performance** — All analysis above uses the full dataset; true out-of-sample testing requires holding back a period (e.g., the last 6 months) and testing on that only
- **Position sizing** — This analysis only measures win rate, not optimal bet size (Kelly criterion or fixed-fraction sizing)

---

### Suggested Next Steps

1. **Update `fixedConLocUpTo03-5-24.xlsx`** — Add contract locations beyond 2024-03-05 to expand the dataset
2. **Out-of-sample test** — Re-run `preparing.ipynb` on a held-out period (e.g., post-March 2024) and check if the combined filter holds
3. **Model transaction costs** — Estimate net P&L per trade given Nadex fee structure and your typical contract size
4. **Paper trade** — Run the filtered strategy rules live on Nadex in paper mode for 30+ days before committing capital